In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e9/train.csv
/kaggle/input/competitions/playground-series-s6e9/test.csv
/kaggle/input/datasets/keyadobriyal/original-dataset/EV_Adoption_and_Range_Anxiety_Dataset.csv


In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings("ignore")
print("Libraries loaded.")


Libraries loaded.


In [3]:
# ============================================================
# 2. DATA PATHS
# ============================================================
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e9/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e9/test.csv'
SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv'
ORIG_PATH  = '/kaggle/input/datasets/keyadobriyal/original-dataset/EV_Adoption_and_Range_Anxiety_Dataset.csv'

# For local execution, change the paths above.

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
orig = pd.read_csv(ORIG_PATH)
submission_template = pd.read_csv(SUB_PATH)

print("Train:", train.shape)
print("Test :", test.shape)
print("Reference/original EV data:", orig.shape)
print(train['Will_Buy_EV'].value_counts(dropna=False))


Train: (668665, 15)
Test : (286571, 14)
Reference/original EV data: (10000, 15)
Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64


In [4]:
# ============================================================
# 3. OPTIMIZED FEATURE ENGINEERING
# ============================================================
TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
orig[TARGET] = orig[TARGET].map({'Yes': 1, 'No': 0})

# Work on a train+test feature frame only for target-free transformations.
# Competition labels are never used here.
train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined.drop(columns=['Number_of_Cars_Owned'], inplace=True, errors='ignore')

# -----------------------------
# A. Identify raw feature types
# -----------------------------
cat_cols = combined.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['id', TARGET, 'is_train']]
num_cols = [c for c in combined.columns if c not in cat_cols + ['id', 'is_train', TARGET]]

# -----------------------------
# B. Missingness indicators
# -----------------------------
# Missingness can carry signal in synthetic tabular data and costs very little.
for c in list(num_cols):
    if combined[c].isna().any():
        combined[f'{c}_is_missing'] = combined[c].isna().astype('int8')

# -----------------------------
# C. Compact digit decomposition
# -----------------------------
# The old pipeline generated 8 digits for EVERY numeric variable. That creates
# many sparse/noisy categorical features. Keep digit decomposition only for
# variables where decimal/integer structure is most likely to matter.
DIGIT_ENGINEER_COLS = [
    c for c in ['Annual_Income_USD', 'Daily_Commute_km', 'Age'] if c in combined.columns
]
DIGIT_K = [-2, -1, 0, 1, 2, 3]
digit_features = []
for c in DIGIT_ENGINEER_COLS:
    values = pd.to_numeric(combined[c], errors='coerce').fillna(0)
    for k in DIGIT_K:
        col_name = f'{c}_digit{k}'
        combined[col_name] = (np.floor(values / (10.0 ** k)) % 10).astype('int8')
        digit_features.append(col_name)

# -----------------------------
# D. Domain features
# -----------------------------
def safe_num(c):
    return pd.to_numeric(combined[c], errors='coerce') if c in combined.columns else None

income = safe_num('Annual_Income_USD')
commute = safe_num('Daily_Commute_km')
env = safe_num('Environmental_Concern_Level')

if income is not None:
    combined['income_log1p'] = np.log1p(income.clip(lower=0))
    combined['income_30k_flag'] = (income == 30000).astype('int8')
    combined['income_high_flag'] = (income >= income.quantile(0.95)).astype('int8')
    combined['income_low_flag'] = (income <= income.quantile(0.05)).astype('int8')

if commute is not None:
    combined['commute_log1p'] = np.log1p(commute.clip(lower=0))
    combined['commute_long_flag'] = (commute >= commute.quantile(0.90)).astype('int8')

if income is not None and commute is not None:
    combined['income_per_commute'] = income / (commute.abs() + 1.0)
    combined['commute_income_ratio'] = commute / (income.abs() / 10000.0 + 1.0)

if env is not None:
    combined['is_env_hater'] = (env == 1).astype('int8')
    combined['env_high_flag'] = (env >= env.quantile(0.75)).astype('int8')

# Preserve the strong empirical income regions from the supplied pipeline,
# but make them explicit/configurable rather than hiding them in pruning.
if income is not None:
    combined['is_30k_spike'] = (income == 30000).astype('int8')
    combined['is_millionaire_cliff'] = (income >= 170537).astype('int8')
    combined['is_dead_zone'] = ((income >= 38000) & (income <= 42000)).astype('int8')

# -----------------------------
# E. Smooth numeric keys
# -----------------------------
# These are retained because they provide a controlled way for trees/TE to
# model local structure without creating dozens of arbitrary bins.
smooth_key_specs = {
    'income_exact_int': ('Annual_Income_USD', 1.0),
    'income100_floor': ('Annual_Income_USD', 100.0),
    'income1000_floor': ('Annual_Income_USD', 1000.0),
    'commute_integer': ('Daily_Commute_km', 1.0),
}
for out, (src, divisor) in smooth_key_specs.items():
    if src in combined.columns:
        vals = pd.to_numeric(combined[src], errors='coerce')
        # Keep engineered keys numeric. High-cardinality keys (e.g. exact income)
        # may intentionally bypass target encoding, but must remain LightGBM/XGBoost
        # compatible. Lower-cardinality keys can still be passed to TargetEncoder.
        combined[out] = (
            np.floor(vals / divisor)
            .fillna(-999999)
            .astype('int64')
        )

# -----------------------------
# F. Reference/original-dataset target statistics
# -----------------------------
# This uses the supplied external/reference dataset, not competition labels.
# Keep it as a separate numeric signal so the model can decide whether it helps.
orig_global_mean = orig[TARGET].mean()
org_mean_sources = [c for c in (cat_cols + num_cols) if c in orig.columns]
for c in org_mean_sources:
    stats = orig.groupby(c, observed=False)[TARGET].mean()
    combined[f'{c}_org_mean'] = combined[c].map(stats).fillna(orig_global_mean).astype('float32')

# -----------------------------
# G. Frequency encoding
# -----------------------------
# Frequency is target-free, so it may be calculated on train+test. Use float32
# to reduce memory pressure.
frequency_cols = cat_cols + [c for c in smooth_key_specs if c in combined.columns]
for c in frequency_cols:
    freq = combined[c].value_counts(dropna=False, normalize=True)
    combined[f'{c}_fe'] = combined[c].map(freq).fillna(0).astype('float32')

# -----------------------------
# H. Compact categorical candidate set for TE
# -----------------------------
# IMPORTANT OPTIMIZATION: do NOT turn every numeric/digit feature into a string
# and triple-TE all of them. This was the main source of feature explosion.
# TE candidates are restricted to genuine categoricals + smooth keys and are
# further filtered by cardinality.
TE_MAX_CARDINALITY = 5000
TE_MIN_FREQUENCY = 2
te_candidates = []
for c in frequency_cols:
    nunique = combined[c].nunique(dropna=False)
    if nunique <= TE_MAX_CARDINALITY:
        te_candidates.append(c)

# Add a small number of digit categories only where they are intentionally built.
for c in digit_features:
    if c in combined.columns:
        te_candidates.append(c)

# Deduplicate while preserving order.
TARGET_ENCODE_COLS = list(dict.fromkeys(te_candidates))

# -----------------------------
# I. Remove constants and near-duplicate numeric features
# -----------------------------
train_fe = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_fe = combined[combined['is_train'] == 0].drop(columns=['is_train', TARGET])

# Constant columns.
to_drop_const = sorted(set(
    [c for c in train_fe.columns if c not in ['id', TARGET] and train_fe[c].nunique(dropna=False) <= 1] +
    [c for c in test_fe.columns if c != 'id' and test_fe[c].nunique(dropna=False) <= 1]
))

# Exact/near-exact duplicate numeric columns. A high threshold removes copies
# while retaining genuinely distinct signals.
eval_cols = [c for c in train_fe.columns if c not in ['id', TARGET] and pd.api.types.is_numeric_dtype(train_fe[c])]
to_drop_corr = []
if len(eval_cols) > 1:
    corr = train_fe[eval_cols].corr(method='pearson').abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop_corr = [c for c in upper.columns if (upper[c] >= 0.99999).any()]

DROP = sorted(set(to_drop_const).union(to_drop_corr))
DROP = [c for c in DROP if c not in ['id', TARGET]]
if DROP:
    print(f'Optimized FE: dropping {len(DROP)} constant/near-duplicate features.')
    train_fe.drop(columns=DROP, inplace=True, errors='ignore')
    test_fe.drop(columns=DROP, inplace=True, errors='ignore')

train = train_fe
# Rebuild test after synchronized feature dropping.
test = test_fe

FEATURES = [c for c in test.columns if c != 'id']
TARGET_ENCODE_COLS = [c for c in TARGET_ENCODE_COLS if c in FEATURES and c not in DROP]

print('\n' + '=' * 65)
print('OPTIMIZED FEATURE ENGINEERING SUMMARY')
print('=' * 65)
print(f'Raw categorical columns       : {len(cat_cols)}')
print(f'Original numeric columns      : {len(num_cols)}')
print(f'Intentional digit columns     : {len(digit_features)}')
print(f'Frequency/TE candidate cols   : {len(frequency_cols)}')
print(f'Target-encoding candidates    : {len(TARGET_ENCODE_COLS)}')
print(f'Final base features           : {len(FEATURES)}')
print(f'Constant/near-duplicate drop  : {len(DROP)}')
print('=' * 65)

# Save an audit table for reproducibility.
feature_audit = pd.DataFrame({'feature': FEATURES})
feature_audit['is_te_candidate'] = feature_audit['feature'].isin(TARGET_ENCODE_COLS)
feature_audit['dtype'] = [str(train[c].dtype) for c in FEATURES]
feature_audit.to_csv('feature_engineering_audit.csv', index=False)


Optimized FE: dropping 14 constant/near-duplicate features.

OPTIMIZED FEATURE ENGINEERING SUMMARY
Raw categorical columns       : 6
Original numeric columns      : 6
Intentional digit columns     : 18
Frequency/TE candidate cols   : 10
Target-encoding candidates    : 18
Final base features           : 55
Constant/near-duplicate drop  : 14


In [5]:
# ============================================================
# 4. LEAN MULTI-SMOOTH TARGET ENCODING
# ============================================================
def triple_target_encode(X_train, y_train, X_valid, X_test, target_encode_cols, folds):
    """Leakage-safe TE with three smoothness levels on a compact candidate set.

    The outer CV split is respected: the encoder is fitted only on X_train/y_train,
    while validation/test rows are transformed with the fitted encoders.
    """
    X_train = X_train.copy()
    X_valid = X_valid.copy()
    X_test = X_test.copy()

    cols = [c for c in target_encode_cols if c in X_train.columns]
    if not cols:
        return X_train, X_valid, X_test

    # Three smoothings are retained, but only for selected categorical candidates.
    smoothings = [('auto', 'TE_auto'), (10.0, 'TE_10'), (100.0, 'TE_100')]
    for smooth, suffix in smoothings:
        te = TargetEncoder(
            shuffle=True,
            cv=folds,
            smooth=smooth,
            random_state=42,
            target_type='binary'
        )
        tr = te.fit_transform(X_train[cols], y_train)
        va = te.transform(X_valid[cols])
        ts = te.transform(X_test[cols])
        for i, col in enumerate(cols):
            X_train[f'{col}_{suffix}'] = tr[:, i].astype('float32')
            X_valid[f'{col}_{suffix}'] = va[:, i].astype('float32')
            X_test[f'{col}_{suffix}'] = ts[:, i].astype('float32')

    # Original categorical columns are not directly used after TE.
    X_train.drop(columns=cols, inplace=True, errors='ignore')
    X_valid.drop(columns=cols, inplace=True, errors='ignore')
    X_test.drop(columns=cols, inplace=True, errors='ignore')

    # Defensive safeguard: any categorical/object column not selected for TE
    # is converted to consistent integer codes using a target-free vocabulary.
    remaining_cat = [
        c for c in X_train.columns
        if str(X_train[c].dtype) in ('object', 'string', 'category')
    ]
    for c in remaining_cat:
        all_vals = pd.concat(
            [X_train[c], X_valid[c], X_test[c]],
            ignore_index=True
        ).astype('string')
        categories = pd.Index(all_vals.dropna().unique())
        mapping = pd.Series(np.arange(len(categories), dtype='int32'),
                            index=categories)
        for df in (X_train, X_valid, X_test):
            df[c] = df[c].astype('string').map(mapping).fillna(-1).astype('int32')

    # Final numeric/bool contract for both LightGBM and XGBoost.
    for df in (X_train, X_valid, X_test):
        bad = [c for c in df.columns if not (
            pd.api.types.is_numeric_dtype(df[c]) or
            pd.api.types.is_bool_dtype(df[c])
        )]
        if bad:
            raise TypeError(
                f'Non-numeric model features remain after target encoding: {bad}'
            )

    return X_train, X_valid, X_test


print("Lean Multi-Smooth Target Encoding Process Completed")

Lean Multi-Smooth Target Encoding Process Completed


In [6]:
# ============================================================
# 5. COMMON MODEL INPUT
# ============================================================
TARGET = 'Will_Buy_EV'
FEATURES = [c for c in test.columns if c != 'id']
TARGET_ENCODE_COLS = [c for c in TARGET_ENCODE_COLS if c in FEATURES and c not in DROP]

X = train[FEATURES]
y = train[TARGET]
X_test_base = test[FEATURES]

print(f"Base features: {len(FEATURES)}")
print(f"Target-encoded columns: {len(TARGET_ENCODE_COLS)}")


Base features: 55
Target-encoded columns: 18


In [7]:
# ============================================================
# 6. LIGHTGBM TRAINING
# ============================================================
LGB_FOLDS = 7
lgb_skf = StratifiedKFold(n_splits=LGB_FOLDS, shuffle=True, random_state=42)

lgb_oof = np.zeros(len(train))
lgb_test = np.zeros(len(test))
lgb_importance = None

print(f"Training LIGHTGBM with {LGB_FOLDS} folds...")

for fold, (train_idx, valid_idx) in enumerate(lgb_skf.split(X, y), 1):
    X_train = X.iloc[train_idx].copy()
    y_train = y.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()
    y_valid = y.iloc[valid_idx].copy()
    X_test_fold = X_test_base.copy()

    X_train, X_valid, X_test_fold = triple_target_encode(
        X_train, y_train, X_valid, X_test_fold,
        TARGET_ENCODE_COLS, LGB_FOLDS
    )

    clf = lgb.LGBMClassifier(
        n_estimators=20000,
        learning_rate=0.02,
        max_depth=5,
        num_leaves=32,
        min_child_samples=10,
        subsample=0.8,
        colsample_bytree=0.3,
        reg_alpha=0.071,
        reg_lambda=2.0,
        max_bin=1024,
        random_state=42,
        feature_pre_filter=False,
        metric='auc',
        n_jobs=-1,
        verbose=-1
    )

    clf.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=500, verbose=False),
            lgb.log_evaluation(period=1000)
        ]
    )

    valid_probs = clf.predict_proba(X_valid)[:, 1]
    lgb_oof[valid_idx] = valid_probs
    lgb_test += clf.predict_proba(X_test_fold)[:, 1] / LGB_FOLDS

    if lgb_importance is None:
        lgb_importance = pd.DataFrame({
            "feature": X_train.columns,
            "importance": 0.0
        })
    lgb_importance["importance"] += clf.feature_importances_ / LGB_FOLDS

    fold_auc = roc_auc_score(y_valid, valid_probs)
    print(f"Fold {fold}: best tree #{clf.best_iteration_} | ROC-AUC={fold_auc:.5f}")

lgb_auc = roc_auc_score(y, lgb_oof)
print(f"LIGHTGBM OOF ROC-AUC: {lgb_auc:.6f}")


Training LIGHTGBM with 7 folds...
[1000]	valid_0's auc: 0.942142
[2000]	valid_0's auc: 0.942879
[3000]	valid_0's auc: 0.943235
[4000]	valid_0's auc: 0.943307
[5000]	valid_0's auc: 0.943346
Fold 1: best tree #4752 | ROC-AUC=0.94336
[1000]	valid_0's auc: 0.94289
[2000]	valid_0's auc: 0.943671
[3000]	valid_0's auc: 0.94391
[4000]	valid_0's auc: 0.943989
Fold 2: best tree #3982 | ROC-AUC=0.94399
[1000]	valid_0's auc: 0.944211
[2000]	valid_0's auc: 0.944852
[3000]	valid_0's auc: 0.945094
[4000]	valid_0's auc: 0.945192
Fold 3: best tree #4297 | ROC-AUC=0.94520
[1000]	valid_0's auc: 0.94418
[2000]	valid_0's auc: 0.944853
[3000]	valid_0's auc: 0.945077
[4000]	valid_0's auc: 0.945195
[5000]	valid_0's auc: 0.945195
Fold 4: best tree #4578 | ROC-AUC=0.94521
[1000]	valid_0's auc: 0.944179
[2000]	valid_0's auc: 0.944734
[3000]	valid_0's auc: 0.944884
[4000]	valid_0's auc: 0.944874
Fold 5: best tree #3622 | ROC-AUC=0.94491
[1000]	valid_0's auc: 0.94393
[2000]	valid_0's auc: 0.94474
[3000]	valid_0's 

In [8]:
# LightGBM output files
pd.DataFrame({"id": train["id"], "OOF_Pred": lgb_oof}).to_csv(
    "oof_LIGHTGBM.csv", index=False
)
pd.DataFrame({"id": test["id"], TARGET: lgb_test}).to_csv(
    "test_LIGHTGBM.csv", index=False
)

sub = submission_template.copy()
sub[TARGET] = lgb_test
sub.to_csv("submission_LIGHTGBM.csv", index=False)

lgb_importance = lgb_importance.sort_values(
    "importance", ascending=False
).reset_index(drop=True)
lgb_importance.to_csv("feature_importance_LIGHTGBM.csv", index=False)

print(lgb_importance.head(15))


                             feature   importance
0                 income100_floor_fe  5641.285714
1                income_exact_int_fe  4588.857143
2                  Annual_Income_USD  4107.714286
3           income1000_floor_TE_auto  3599.857143
4             income1000_floor_TE_10  3205.857143
5                       income_log1p  3034.285714
6            income1000_floor_TE_100  3029.571429
7                 income_per_commute  3005.142857
8               commute_income_ratio  2817.142857
9            commute_integer_TE_auto  2662.857143
10  Annual_Income_USD_digit2_TE_auto  2462.571429
11  Annual_Income_USD_digit0_TE_auto  2340.714286
12                               Age  2246.857143
13         Daily_Commute_km_org_mean  2228.285714
14  Annual_Income_USD_digit1_TE_auto  2225.714286


In [9]:
# ============================================================
# 8. XGBOOST TRAINING
# ============================================================

GAIN_COVERAGE = 0.995       # retain features covering this fraction of total gain
MIN_FEATURES = 15           # safety floor so pruning cannot become too aggressive
MAX_FEATURES = None         # set an integer to impose an optional upper cap
RANK_VALID_SIZE = 0.20      # internal split used ONLY for the ranking model
RANK_RANDOM_STATE = 1000
RANK_N_ESTIMATORS = 3000
RANK_EARLY_STOPPING = 100


def xgb_params(n_estimators=100_000, early_stopping_rounds=500):
    return dict(
        n_estimators=n_estimators,
        learning_rate=0.01,
        max_depth=4,
        min_child_weight=4.532387806880492,
        subsample=0.7400402414525654,
        colsample_bytree=0.5695776529558766,
        alpha=0.7523885021652775,
        reg_lambda=0.6189949691705282,
        max_bin=1024,
        random_state=42,
        eval_metric='auc',
        early_stopping_rounds=early_stopping_rounds,
        tree_method='hist',
        n_jobs=-1,
        booster='gbtree',
        objective='binary:logistic',
        max_leaves=16,
        grow_policy='lossguide',
        gamma=3.673225596759869
    )


def select_features_by_gain(X_fit, y_fit, coverage=GAIN_COVERAGE,
                             min_features=MIN_FEATURES, max_features=MAX_FEATURES,
                             fold=1):
    """Learn a fold-specific feature subset using training data only.

    The ranking model is fitted on an internal split of the outer training
    partition. Its gain is therefore never computed using the outer validation
    labels. Features are retained until cumulative gain reaches `coverage`.
    """
    rank_train_idx, rank_valid_idx = train_test_split(
        np.arange(len(X_fit)),
        test_size=RANK_VALID_SIZE,
        stratify=y_fit,
        random_state=RANK_RANDOM_STATE + fold
    )

    rank_model = xgb.XGBClassifier(**xgb_params(
        n_estimators=RANK_N_ESTIMATORS,
        early_stopping_rounds=RANK_EARLY_STOPPING
    ))

    rank_model.fit(
        X_fit.iloc[rank_train_idx], y_fit.iloc[rank_train_idx],
        eval_set=[(X_fit.iloc[rank_valid_idx], y_fit.iloc[rank_valid_idx])],
        verbose=False
    )

    booster = rank_model.get_booster()
    gain_dict = booster.get_score(importance_type='gain')
    gain = pd.Series(0.0, index=X_fit.columns, dtype=float)
    for feature_name, value in gain_dict.items():
        if feature_name in gain.index:
            gain.loc[feature_name] = float(value)

    gain = gain.sort_values(ascending=False)
    positive_gain = gain[gain > 0]

    if positive_gain.empty:
        selected = list(gain.head(min(min_features, len(gain))).index)
    else:
        total_gain = positive_gain.sum()
        cumulative = positive_gain.cumsum() / total_gain
        selected = list(cumulative[cumulative <= coverage].index)
        # Include the first feature that crosses the coverage threshold.
        if len(selected) < len(positive_gain):
            selected.append(positive_gain.index[len(selected)])
        # Enforce the minimum feature count.
        if len(selected) < min_features:
            selected = list(positive_gain.head(min(min_features, len(positive_gain))).index)

    if max_features is not None and len(selected) > max_features:
        selected = list(positive_gain.head(max_features).index)

    # Preserve model-column order for reproducibility.
    selected = [c for c in X_fit.columns if c in set(selected)]
    dropped = [c for c in X_fit.columns if c not in set(selected)]

    ranking = gain.reset_index()
    ranking.columns = ['feature', 'gain']
    ranking['gain_fraction'] = ranking['gain'] / max(total_gain if positive_gain.size else 1.0, 1e-12)
    ranking['selected'] = ranking['feature'].isin(selected)
    ranking['fold'] = fold

    print(
        f"Fold {fold}: ranking model best tree #{rank_model.best_iteration}; "
        f"selected {len(selected)}/{len(X_fit.columns)} features "
        f"({len(dropped)} pruned), cumulative gain target={coverage:.3f}"
    )

    return selected, ranking

print(f"Automatic gain coverage target: {GAIN_COVERAGE:.3f}")
print(f"Minimum retained features: {MIN_FEATURES}")


Automatic gain coverage target: 0.995
Minimum retained features: 15


In [10]:
# ============================================================
# 8. XGBOOST TRAINING WITH FOLD-BY-FOLD AUTOMATIC GAIN PRUNING
# ============================================================
XGB_FOLDS = 5
xgb_skf = StratifiedKFold(n_splits=XGB_FOLDS, shuffle=True, random_state=42)

xgb_oof = np.zeros(len(train))
xgb_test = np.zeros(len(test))
xgb_importance_accum = {}
pruning_rows = []
all_gain_rankings = []

print(f"Training XGBOOST with {XGB_FOLDS} folds and automatic gain pruning...")

for fold, (train_idx, valid_idx) in enumerate(xgb_skf.split(X, y), 1):
    X_train = X.iloc[train_idx].copy()
    y_train = y.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()
    y_valid = y.iloc[valid_idx].copy()
    X_test_fold = X_test_base.copy()

    # Target encoding is fitted using this outer training partition only.
    X_train, X_valid, X_test_fold = triple_target_encode(
        X_train, y_train, X_valid, X_test_fold,
        TARGET_ENCODE_COLS, XGB_FOLDS
    )

    # ------------------------------------------------------------
    # Genuine fold-specific feature selection.
    # The outer validation set is NOT used for ranking/pruning.
    # ------------------------------------------------------------
    selected_features, gain_ranking = select_features_by_gain(
        X_train, y_train, fold=fold
    )
    all_gain_rankings.append(gain_ranking)

    X_train_sel = X_train[selected_features]
    X_valid_sel = X_valid[selected_features]
    X_test_sel = X_test_fold[selected_features]

    pruning_rows.append({
        'fold': fold,
        'total_features': X_train.shape[1],
        'selected_features': len(selected_features),
        'pruned_features': X_train.shape[1] - len(selected_features),
        'retention_pct': 100.0 * len(selected_features) / X_train.shape[1]
    })

    clf = xgb.XGBClassifier(**xgb_params())

    clf.fit(
        X_train_sel, y_train,
        eval_set=[(X_valid_sel, y_valid)],
        verbose=1000
    )

    # Accumulate final-model gain importance across folds. Missing features
    # in a fold receive zero importance.
    final_gain = pd.Series(0.0, index=selected_features, dtype=float)
    booster_gain = clf.get_booster().get_score(importance_type='gain')
    for feature_name, value in booster_gain.items():
        if feature_name in final_gain.index:
            final_gain.loc[feature_name] = float(value)
    for feature_name, value in final_gain.items():
        xgb_importance_accum[feature_name] = xgb_importance_accum.get(feature_name, 0.0) + value / XGB_FOLDS

    valid_probs = clf.predict_proba(X_valid_sel)[:, 1]
    xgb_oof[valid_idx] = valid_probs
    xgb_test += clf.predict_proba(X_test_sel)[:, 1] / XGB_FOLDS

    fold_auc = roc_auc_score(y_valid, valid_probs)
    print(
        f"Fold {fold}: best tree #{clf.best_iteration} | "
        f"features={len(selected_features)} | ROC-AUC={fold_auc:.5f}"
    )

xgb_auc = roc_auc_score(y, xgb_oof)
xgb_importance = pd.DataFrame({
    'feature': list(xgb_importance_accum.keys()),
    'importance': list(xgb_importance_accum.values())
}).sort_values('importance', ascending=False).reset_index(drop=True)

pruning_summary = pd.DataFrame(pruning_rows)
gain_rankings = pd.concat(all_gain_rankings, ignore_index=True)

print("\n" + "=" * 65)
print("XGBOOST AUTOMATIC GAIN-PRUNING SUMMARY")
print("=" * 65)
print(pruning_summary.to_string(index=False))
print(f"\nMean retained features: {pruning_summary['selected_features'].mean():.1f}")
print(f"XGBOOST OOF ROC-AUC: {xgb_auc:.6f}")


Training XGBOOST with 5 folds and automatic gain pruning...
Fold 1: ranking model best tree #2998; selected 77/91 features (14 pruned), cumulative gain target=0.995
[0]	validation_0-auc:0.92394
[1000]	validation_0-auc:0.94127
[2000]	validation_0-auc:0.94217
[3000]	validation_0-auc:0.94259
[4000]	validation_0-auc:0.94280
[5000]	validation_0-auc:0.94292
[6000]	validation_0-auc:0.94299
[7000]	validation_0-auc:0.94305
[8000]	validation_0-auc:0.94309
[9000]	validation_0-auc:0.94313
[10000]	validation_0-auc:0.94316
[11000]	validation_0-auc:0.94318
[12000]	validation_0-auc:0.94321
[13000]	validation_0-auc:0.94322
[14000]	validation_0-auc:0.94325
[15000]	validation_0-auc:0.94327
[16000]	validation_0-auc:0.94328
[17000]	validation_0-auc:0.94329
[18000]	validation_0-auc:0.94329
[19000]	validation_0-auc:0.94331
[20000]	validation_0-auc:0.94331
[21000]	validation_0-auc:0.94332
[22000]	validation_0-auc:0.94333
[23000]	validation_0-auc:0.94334
[24000]	validation_0-auc:0.94334
[25000]	validation_0-au

In [11]:
# XGBoost output files
pd.DataFrame({"id": train["id"], "OOF_Pred": xgb_oof}).to_csv(
    "oof_XGBOOST.csv", index=False
)
pd.DataFrame({"id": test["id"], TARGET: xgb_test}).to_csv(
    "test_XGBOOST.csv", index=False
)

sub = submission_template.copy()
sub[TARGET] = xgb_test
sub.to_csv("submission_XGBOOST.csv", index=False)

xgb_importance.to_csv("feature_importance_XGBOOST.csv", index=False)
pruning_summary.to_csv("xgboost_auto_gain_pruning_summary.csv", index=False)
gain_rankings.to_csv("xgboost_fold_gain_rankings.csv", index=False)

print("Final XGBoost feature importance:")
print(xgb_importance.head(15))
print("\nFold-by-fold pruning summary:")
display(pruning_summary)


Final XGBoost feature importance:
                                 feature   importance
0                          env_high_flag  3279.689941
1             Subsidy_Available_org_mean  2476.526807
2              Subsidy_Available_TE_auto  1557.106934
3            Environmental_Concern_Level  1390.884766
4   Environmental_Concern_Level_org_mean  1294.080640
5               Subsidy_Available_TE_100  1268.548694
6                Subsidy_Available_TE_10  1232.213184
7                           is_env_hater   610.685291
8                 Range_Anxiety_Level_fe   230.203137
9           Range_Anxiety_Level_org_mean   222.832016
10              income1000_floor_TE_auto   128.148856
11                income1000_floor_TE_10   112.413023
12               income1000_floor_TE_100    86.087713
13           Range_Anxiety_Level_TE_auto    63.423764
14            Range_Anxiety_Level_TE_100    56.392505

Fold-by-fold pruning summary:


,fold,total_features,selected_features,pruned_features,retention_pct
0,1,91,77,14,84.615385
1,2,91,78,13,85.714286
2,3,91,78,13,85.714286
3,4,91,78,13,85.714286
4,5,91,79,12,86.813187


In [12]:
# ============================================================
# 9. OOF BLENDING
# ============================================================
blend_rows = []

for w in np.linspace(0.0, 1.0, 101):
    pred = w * xgb_oof + (1.0 - w) * lgb_oof
    blend_rows.append((w, roc_auc_score(y, pred)))

blend_results = pd.DataFrame(
    blend_rows, columns=["xgb_weight", "oof_auc"]
)

best_row = blend_results.loc[blend_results["oof_auc"].idxmax()]
best_xgb_weight = float(best_row["xgb_weight"])
best_lgb_weight = 1.0 - best_xgb_weight

ensemble_oof = (
    best_xgb_weight * xgb_oof
    + best_lgb_weight * lgb_oof
)
ensemble_auc = roc_auc_score(y, ensemble_oof)

print("=" * 65)
print("MODEL COMPARISON")
print("=" * 65)
print(f"LightGBM OOF ROC-AUC : {lgb_auc:.6f}")
print(f"XGBoost OOF ROC-AUC  : {xgb_auc:.6f}")
print(f"XGBoost blend weight : {best_xgb_weight:.2f}")
print(f"LightGBM blend weight: {best_lgb_weight:.2f}")
print(f"Ensemble OOF ROC-AUC : {ensemble_auc:.6f}")
print("=" * 65)

display(blend_results.sort_values("oof_auc", ascending=False).head(10))


MODEL COMPARISON
LightGBM OOF ROC-AUC : 0.944611
XGBoost OOF ROC-AUC  : 0.944474
XGBoost blend weight : 0.37
LightGBM blend weight: 0.63
Ensemble OOF ROC-AUC : 0.944684


,xgb_weight,oof_auc
37,0.37,0.944684
38,0.38,0.944684
36,0.36,0.944684
35,0.35,0.944683
39,0.39,0.944683
34,0.34,0.944683
40,0.40,0.944683
33,0.33,0.944683
41,0.41,0.944683
32,0.32,0.944682


In [13]:
# ============================================================
# 10. FINAL ENSEMBLE SUBMISSION
# ============================================================
ensemble_test = (
    best_xgb_weight * xgb_test
    + best_lgb_weight * lgb_test
)

final_submission = submission_template.copy()
final_submission[TARGET] = ensemble_test
final_submission.to_csv("submission.csv", index=False)

pd.DataFrame({
    "id": train["id"],
    TARGET: y,
    "LGBM_OOF": lgb_oof,
    "XGB_OOF": xgb_oof,
    "ENSEMBLE_OOF": ensemble_oof
}).to_csv(
    "oof_EV_PURCHASE_ENSEMBLE.csv",
    index=False
)

pd.DataFrame({
    "id": test["id"],
    "LGBM_Pred": lgb_test,
    "XGB_Pred": xgb_test,
    "Ensemble_Pred": ensemble_test
}).to_csv(
    "test_EV_PURCHASE_ENSEMBLE.csv",
    index=False
)

blend_results.to_csv("blend_weight_search.csv", index=False)

print("Final submission:", final_submission.shape)
print(final_submission.head())
print("\nPrediction statistics:")
print(pd.Series(ensemble_test).describe())


Final submission: (286571, 2)
       id  Will_Buy_EV
0  668665     0.014840
1  668666     0.014706
2  668667     0.004258
3  668668     0.002148
4  668669     0.025111

Prediction statistics:
count    286571.000000
mean          0.175729
std           0.274238
min           0.000002
25%           0.002351
50%           0.017226
75%           0.263980
max           0.999607
dtype: float64
